# FLUX.2 Klein 4B NVFP4 Inference

Colab notebook for runtime NVFP4 inference with Diffusers, TorchAO, and the Klein pipeline.

This follows the same runtime path as `modal_bench_inference_nvfp4.py`:
- install deps
- clone the Klein repo
- download the model
- load NVFP4 weights at runtime
- run image generation

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-U', *packages]
    subprocess.check_call(cmd)

pip_install(
    'torch',
    'torchvision',
    'accelerate',
    'safetensors',
    'huggingface_hub',
    'transformers',
    'einops',
    'pillow',
    'cache-dit',
    'diffusers',
    'torchao',
)

try:
    pip_install('mslk-cuda')
    print('mslk-cuda installed')
except Exception as exc:
    print('mslk-cuda install failed, Triton NVFP4 may fall back or fail:', exc)

repo_dir = Path('/content/klein4B')
if not repo_dir.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Yuvrajxms09/klein4B.git', str(repo_dir)])

sys.path.insert(0, str(repo_dir))

print('setup complete')

In [ ]:
from huggingface_hub import snapshot_download

MODEL_REPO = 'black-forest-labs/FLUX.2-klein-4B'
MODEL_DIR = '/content/FLUX.2-klein-4B'

model_dir = snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)
print('model_dir=', model_dir)

In [ ]:
import torch
from PIL import Image
from diffusers import Flux2Transformer2DModel, TorchAoConfig
from torchao.prototype.mx_formats import NVFP4DynamicActivationNVFP4WeightConfig

from klein_pipeline import Flux2KleinPipeline
from cache_dit_klein import enable_cache_dit, prepare_transformer_for_speed

torch.set_grad_enabled(False)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

quantization_config = TorchAoConfig(
    NVFP4DynamicActivationNVFP4WeightConfig(
        use_triton_kernel=True,
        use_dynamic_per_tensor_scale=True,
    )
)

transformer = Flux2Transformer2DModel.from_pretrained(
    model_dir,
    subfolder='transformer',
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)

pipe = Flux2KleinPipeline.from_pretrained(
    model_dir,
    transformer=transformer,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
pipe = pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)

enable_cache_dit(pipe)
backend = prepare_transformer_for_speed(pipe, backend='auto', fuse_qkv=True)
print('attention backend:', backend)

if hasattr(pipe.vae, 'to'):
    pipe.vae.to(memory_format=torch.channels_last)

print('pipeline ready')

In [ ]:
image_path = '/content/klein4B/blue_car_resize.jpeg'
if not Path(image_path).exists():
    image_path = '/content/klein4B/blue_car.jpeg'

img = Image.open(image_path).convert('RGB')

prompt = 'A blue car driving on a mountain road at sunset'
generator = torch.Generator(device='cuda').manual_seed(0)

with torch.inference_mode():
    out = pipe(
        prompt=prompt,
        image=img,
        height=576,
        width=384,
        num_inference_steps=4,
        guidance_scale=1.0,
        generator=generator,
        output_type='pil',
        return_dict=True,
    )

result = out.images[0]
display(result)
result.save('/content/klein4b_nvfp4_output.png')
print('saved to /content/klein4b_nvfp4_output.png')